<center>
    <font size="5"> Sieci neuronowe i uczenie głębokie<br/>
        <small><em>Studia stacjonarne II stopnia 2025/2026</em><br/>Kierunek: Matematyka stosowana<br>Specjalność: Analityka danych</small>
    </font>
</center>
<br>


# Laboratorium nr 13: Reinforcement learning

## Instalacja i Import bibliotek

!pip install "gymnasium[classic-control]"

In [6]:
# import numpy as np
import matplotlib.pyplot as plt
# import tensorflow as tf
# from tensorflow.keras import layers
import gymnasium as gym
import numpy as np
import imageio
from IPython.display import Image, display
import random
from collections import deque
import keras
print('Numpy version:', np.__version__)
print('Tensorflow version:', keras.__version__)

ImportError: cannot load module more than once per process

ImportError: _multiarray_umath failed to import

ImportError: numpy._core.umath failed to import

### Definicja środowiska

"CartPole-v1" z biblioteki `gym` od OpenAI, [OpenAI Gym](https://gym.openai.com/)

In [ ]:
env = gym.make("CartPole-v1")
env.reset(seed=41) # w celu powtarzalności wyników

### Observations:

1. position of cart
2. velocity of cart
3. angle of pole
4. rotation rate of pole

In [ ]:
print("Observation space = {}".format(env.observation_space))

### Actions

In [ ]:
n_actions = env.action_space.n
print("Number of possible actions = {}".format(n_actions))

## Część 1: Policy Gradient (REINFORCE)
W tej metodzie sieć neuronowa uczy się bezpośrednio polityki $\pi(a|s)$, czyli zwraca prawdopodobieństwo wyboru poszczególnych akcji w danym stanie. W problemie CartPole mamy 2 akcje (w lewo, w prawo), więc na wyjściu sieci używamy funkcji `softmax`.

In [ ]:
# Sieć przewiduje prawdopodobieństwa akcji
def build_policy_network(state_dim, n_actions):
    model = tf.keras.Sequential([
        layers.InputLayer(input_shape=(state_dim,)),
        layers.Dense(32, activation='relu'),
        layers.Dense(32, activation='relu'),
        # Zauważ: 'softmax' sprawia, że wyjście sumuje się do 1 (prawdopodobieństwa)
        layers.Dense(n_actions, activation='softmax') 
    ])
    return model

policy_model = build_policy_network(4, env.action_space.n)

### Funkcja podejmowania wyboru akcji

In [ ]:
def choose_action_pg(model, state):
    # Wybiera akcję stochastycznie na podstawie rozkładu prawdopodobieństwa z modelu.
    state = state.reshape([1, -1])
    prob_weights = model.predict(state, verbose=0)[0]
    
    # Wybieramy akcję losowo, ale z wagami przewidzianymi przez sieć
    action = np.random.choice(len(prob_weights), p=prob_weights)
    return action

### Agent's memory
Klasa pomocnicza do przechowywania infromacji

In [ ]:
class Memory:
  def __init__(self):
      self.clear()

  def clear(self):
      self.observations = []
      self.actions = []
      self.rewards = []

  def add_to_memory(self, new_observation, new_action, new_reward):
      self.observations.append(new_observation)
      self.actions.append(new_action)
      self.rewards.append(new_reward)

memory = Memory()

def normalize(x):
  x -= np.mean(x)
  x /= np.std(x)
  return x

def discount_rewards(rewards, gamma=0.95):
  discounted_rewards = np.zeros_like(rewards)
  R = 0
  for t in reversed(range(0, len(rewards))):
      # update the total discounted reward
      R = R * gamma + rewards[t]
      discounted_rewards[t] = R
  return normalize(discounted_rewards)

### Jak działa uczenie Policy Gradient w Keras?
Algorytm REINFORCE dąży do maksymalizacji oczekiwanej nagrody. Funkcja straty w tym algorytmie to:
$$L = - \sum \log(\pi(a_t|s_t)) \cdot R_t$$
Gdzie:
- $\log(\pi(a_t|s_t))$ to logarytm prawdopodobieństwa podjęcia wybranej akcji $a_t$ w stanie $s_t$.
- $R_t$ to zdyskontowana nagroda (suma przyszłych nagród od momentu $t$).

Sztuczka Keras: Jako funkcję straty podajemy standardowe `sparse_categorical_crossentropy`. W normalnej klasyfikacji funkcja ta zachęca model do zwracania 1.0 dla "prawdziwej" klasy. W RL podajemy podjętą akcję jako "prawdziwą klasę", ale używamy parametru `sample_weight=discount_rewards`. Dzięki temu, jeśli nagroda $R_t$ była dodatnia (dobry ruch), waga jest duża i model jest mocno zachęcany do powtórzenia akcji. Jeśli $R_t$ była ujemna/niska, waga karze ten wybór.


### Algorytm uczenia

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
policy_model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy')

EPOCHS_PG = 100
for episode in range(EPOCHS_PG):
    state = env.reset()[0]
    memory.clear()
    
    while True:
        action = choose_action_pg(policy_model, state)
        next_state, reward, done, _, _ = env.step(action)
        
        memory.add_to_memory(state, action, reward)
        
        if done:
            total_reward = sum(memory.rewards)
            
            # 1. Obliczamy zdyskontowane nagrody (R_t)
            discounted_r = discount_rewards(memory.rewards)
            
            # 2. Uczymy model! 
            # X = zapamiętane stany
            # Y = podjęte akcje (traktujemy je chwilowo jako Ground Truth)
            # Wagi = zdyskontowane nagrody (wzmacniają dobre akcje, osłabiają złe)
            policy_model.fit(
                np.vstack(memory.observations), 
                np.vstack(memory.actions), 
                epochs=1, 
                verbose=0,
                sample_weight=discounted_r
            )
            print(f"Epoka: {episode} | Nagroda całkowita: {total_reward}")
            break
            
        state = next_state

In [ ]:
def show_trained_agent_gif(model, env_name="CartPole-v1", filename="trained_agent.gif"):
    """
    Uruchamia jedną grę z wytrenowanym modelem, nagrywa klatki i wyświetla jako GIF.
    """
    print(f"Uruchamianie środowiska {env_name} w celu nagrania...")
    
    # render_mode="rgb_array" pozwala nam wyciągać klatki jako tablice pikseli
    env = gym.make(env_name, render_mode="rgb_array")
    state, _ = env.reset()
    frames = []
    
    done = False
    truncated = False
    
    while not (done or truncated):
        # 1. Zapisanie obecnego obrazu ze środowiska do listy
        frames.append(env.render())
        
        # 2. Wybór akcji. 
        # UWAGA: W trybie TESTOWYM nie chcemy już losowości (eksploracji). 
        # Wybieramy zawsze akcję z najwyższym prawdopodobieństwem (PG) lub Q-value (DQN).
        # Dlatego używamy po prostu funkcji argmax.
        state_batched = state.reshape([1, -1])
        predictions = model.predict(state_batched, verbose=0)
        action = np.argmax(predictions[0])
        
        # 3. Wykonanie kroku w środowisku
        state, reward, done, truncated, _ = env.step(action)
        
    env.close()
    
    # 4. Zapisanie zebranych klatek jako plik GIF
    print(f"Koniec symulacji. Zebrano {len(frames)} klatek. Trwa generowanie pliku GIF...")
    imageio.mimsave(filename, frames, fps=30)
    print(f"Gotowe! Zapisano jako: {filename}")
    
    # 5. Bezpośrednie wyświetlenie animacji w komórce
    with open(filename, "rb") as f:
        display(Image(data=f.read(), format='png'))


show_trained_agent_gif(policy_model, filename="agent_pg.gif")

## Część 2: Deep Q-Learning (DQL / DQN)
W przeciwieństwie do Policy Gradient, algorytmy typu Value-Based (jak DQN) nie próbują zgadnąć od razu "najlepszej akcji". Zamiast tego próbują ocenić jak dobry jest dany stan i akcja. 

Sieć neuronowa w DQN to aproksymator funkcji $Q(s, a)$. Zwraca ona przewidywaną, zdyskontowaną sumę nagród, jaką zdobędziemy do końca gry, jeśli w stanie $s$ wykonamy akcję $a$. Ponieważ nie znamy przyszłości, uczymy sieć iteracyjnie na podstawie Równania Bellmana:
$$Q_{target}(s, a) = r + \gamma \max_{a'} Q(s', a')$$

Kluczowe mechanizmy DQN:
- Brak aktywacji softmax na wyjściu: Sieć przewiduje konkretne wartości liczbowe (Q-values) dla każdej akcji (aktywacja liniowa).
- Epsilon-Greedy: Algorytm musi eksplorować środowisko. Z prawdopodobieństwem $\epsilon$ wybiera akcję losową, a w przeciwnym razie wybiera akcję o najwyższej wartości Q (zachłannie).
- Replay Buffer (Bufor powtórek): Nie uczymy sieci na kolejnych krokach z jednej gry (są zbyt ze sobą skorelowane). Zapamiętujemy tysiące przejść $(s, a, r, s', done)$ i losujemy z nich małe paczki (mini-batches) do uczenia.

### Architektura DQL i pamięć doświadczeń

In [ ]:
#  Definicja modelu Q-Network
def build_dqn_network(state_dim, n_actions):
    model = tf.keras.Sequential([
        layers.InputLayer(input_shape=(state_dim,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(64, activation='relu'),
        # Zauważ aktywację 'linear'! Przewidujemy wartość punktową (Q-value)
        layers.Dense(n_actions, activation='linear') 
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    return model

dqn_model = build_dqn_network(4, env.action_space.n)

# Bufor powtórek (Replay Buffer)
# Używamy deque, aby po przekroczeniu 2000 elementów automatycznie usuwało najstarsze
replay_buffer = deque(maxlen=2000)

def add_to_buffer(state, action, reward, next_state, done):
    replay_buffer.append((state, action, reward, next_state, done))

### Strategia Epsilon-Greedy

In [ ]:
# Zmienne globalne dla eksploracji
epsilon = 1.0       # Zaczynamy od 100% losowych ruchów
epsilon_min = 0.01  # Zawsze zostawiamy 1% losowości
epsilon_decay = 0.995 # Jak szybko zmniejszamy losowość po każdym odcinku

def choose_action_dqn(state, epsilon):
    """Strategia epsilon-greedy: eksploracja vs eksploatacja"""
    if np.random.rand() <= epsilon:
        # Eksploracja: losowa akcja
        return env.action_space.sample()
    
    # Eksploatacja: wybieramy akcję z najwyższym Q-value
    state_batched = state.reshape([1, -1])
    q_values = dqn_model.predict(state_batched, verbose=0)
    return np.argmax(q_values[0])

### Pętla ucząca DQN (Odpytywanie bufora i równanie Bellmana)
To jest serce algorytmu DQL, które wyraźnie różni się od Policy Gradient

In [ ]:
def train_dqn(batch_size=32, gamma=0.95):
    """Pobiera próbki z bufora i wykonuje krok uczenia na podstawie równania Bellmana"""
    if len(replay_buffer) < batch_size:
        return # Nie uczymy, dopóki nie zbierzemy wystarczającej liczby danych
    
    # Losowanie mini-batcha z bufora
    minibatch = random.sample(replay_buffer, batch_size)
    
    # Rozpakowanie batcha do tablic numpy dla szybszego przetwarzania
    states = np.array([transition[0] for transition in minibatch])
    actions = np.array([transition[1] for transition in minibatch])
    rewards = np.array([transition[2] for transition in minibatch])
    next_states = np.array([transition[3] for transition in minibatch])
    dones = np.array([transition[4] for transition in minibatch])

    # Krok 1: Przewidywanie obecnych Q-values dla stanu s (do optymalizacji)
    target_q_values = dqn_model.predict(states, verbose=0)
    
    # Krok 2: Przewidywanie Q-values dla przyszłego stanu s'
    future_q_values = dqn_model.predict(next_states, verbose=0)

    # Krok 3: Równanie Bellmana - aktualizacja wartości docelowych (Targets)
    for i in range(batch_size):
        if dones[i]:
            # Jeśli gra się skończyła, maksymalna przyszła nagroda to po prostu obecna nagroda
            target_q_values[i][actions[i]] = rewards[i]
        else:
            # W przeciwnym wypadku: nagroda + zdyskontowana najlepsza wartość w przyszłości
            target_q_values[i][actions[i]] = rewards[i] + gamma * np.amax(future_q_values[i])

    # Krok 4: Uczenie modelu! 
    # (wejście to stany, wyjście to zaktualizowane Q-values oparte na równaniu Bellmana)
    dqn_model.fit(states, target_q_values, epochs=1, verbose=0)

GŁÓWNA PĘTLA URUCHOMIENIOWA DQN

In [ ]:
EPOCHS_DQN = 100

for episode in range(EPOCHS_DQN):
    state = env.reset()[0]
    total_reward = 0
    
    while True:
        action = choose_action_dqn(state, epsilon)
        next_state, reward, done, _, _ = env.step(action)
        
        # Modyfikacja nagrody pomaga w nauce (kary za porażkę w CartPole)
        reward = reward if not done or total_reward == 499 else -10 
        
        add_to_buffer(state, action, reward, next_state, done)
        state = next_state
        total_reward += 1
        
        # Uczymy agenta na każdym kroku (lub co określoną liczbę kroków)
        train_dqn(batch_size=32)
        
        if done:
            print(f"DQN Epoka: {episode} | Nagroda całkowita: {total_reward} | Epsilon: {epsilon:.2f}")
            break
            
    # Zmniejszanie losowości wraz z postępem uczenia
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

In [ ]:
show_trained_agent_gif(dqn_model, filename="agent_dqn.gif")

## Ćwiczenie
Korzystając ze źródła: https://keras.io/examples/rl/ppo_cartpole/ przeanalizuj i wytrenuj agenta dla cartpole z wykorzystaniem algorytmu PPO